# Summer Analytics 2026 — E-Commerce Conversion Prediction
## Solution Notebook

**Approach:** Ensemble of XGBoost + LightGBM + Random Forest with feature engineering and threshold tuning.

**Evaluation Metric:** F1 Score on private test set.


### Step 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.impute import SimpleImputer
from sklearn.metrics import f1_score, classification_report
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully.")

### Step 2: Load Data

In [ ]:
train = pd.read_csv('train.csv')
pub   = pd.read_csv('public_test.csv')
priv  = pd.read_csv('private_test.csv')

print("Train shape:          ", train.shape)
print("Public test shape:    ", pub.shape)
print("Private test shape:   ", priv.shape)

print("\nTarget distribution (train):")
print(train['Converted'].value_counts(normalize=True).rename({0:'Not Converted', 1:'Converted'}))

print("\nMissing values (train):")
print(train.isnull().sum()[train.isnull().sum() > 0])

train.head()

### Step 3: Exploratory Data Analysis

In [ ]:
print("Device_Type distribution:")
print(train.groupby('Device_Type')['Converted'].mean().sort_values(ascending=False))

print("\nTraffic_Source conversion rate:")
print(train.groupby('Traffic_Source')['Converted'].mean().sort_values(ascending=False))

print("\nCity_Tier conversion rate:")
print(train.groupby('City_Tier')['Converted'].mean().sort_values(ascending=False))

print("\nCorrelation with target (numeric features):")
numeric_cols = ['Age', 'Income', 'Pages_Viewed', 'Products_Viewed',
                'Time_On_Site', 'Previous_Purchases', 'Discount_Seen']
corr = train[numeric_cols + ['Converted']].corr()['Converted'].drop('Converted').sort_values(ascending=False)
print(corr)

### Step 4: Feature Engineering

We engineer several interaction and aggregation features to capture richer signals:
- **Engagement features**: Combining pages viewed, products viewed, and time on site
- **Economic features**: Income per age, high-value user indicator
- **Behavioral features**: Purchase × discount interaction, channel flags
- **Encoded categoricals**: Device type and traffic source mapped to ordinal values

In [ ]:
def engineer_features(df):
    df = df.copy()

    # Encode categorical columns
    df['Device_Type_enc'] = df['Device_Type'].map({'Mobile': 0, 'Tablet': 1, 'Desktop': 2})
    df['Traffic_Source_enc'] = df['Traffic_Source'].map({
        'Organic': 0, 'Referral': 1, 'Social Media': 2, 'Paid Ads': 3, 'Email': 4
    })

    # Engagement interaction features
    df['Pages_x_Time']      = df['Pages_Viewed'] * df['Time_On_Site']
    df['Products_x_Time']   = df['Products_Viewed'] * df['Time_On_Site']
    df['Pages_per_Product'] = df['Pages_Viewed'] / (df['Products_Viewed'] + 1)
    df['Engagement_Score']  = df['Pages_Viewed'] + df['Products_Viewed'] * 2 + df['Time_On_Site'] / 10

    # Economic features
    df['Income_per_Age']    = df['Income'] / (df['Age'] + 1)
    df['High_Value']        = ((df['Income'] > df['Income'].median()) &
                               (df['Previous_Purchases'] > 0)).astype(int)

    # Behavioral features
    df['Purchase_x_Discount'] = df['Previous_Purchases'] * df['Discount_Seen']
    df['Is_Mobile']           = (df['Device_Type'] == 'Mobile').astype(int)
    df['Is_Email']            = (df['Traffic_Source'] == 'Email').astype(int)

    # Age grouping
    df['Age_Group'] = pd.cut(df['Age'], bins=[0, 25, 35, 50, 65, 100],
                             labels=[0, 1, 2, 3, 4]).astype(float)

    return df

# Apply to all splits together (preserving median from full data)
all_data = pd.concat([train, pub, priv], ignore_index=True)
all_eng  = engineer_features(all_data)

FEATURES = [
    'Age', 'Income', 'City_Tier', 'Device_Type_enc', 'Traffic_Source_enc',
    'Pages_Viewed', 'Products_Viewed', 'Time_On_Site', 'Previous_Purchases',
    'Discount_Seen', 'Browser_Version', 'Campaign_Code',
    'Pages_x_Time', 'Products_x_Time', 'Pages_per_Product',
    'Engagement_Score', 'Income_per_Age', 'Purchase_x_Discount',
    'High_Value', 'Is_Mobile', 'Is_Email', 'Age_Group'
]

n_train = len(train)
n_pub   = len(pub)

X_train = all_eng.iloc[:n_train][FEATURES]
y_train = train['Converted']
X_pub   = all_eng.iloc[n_train:n_train+n_pub][FEATURES]
y_pub   = pub['Converted']
X_priv  = all_eng.iloc[n_train+n_pub:][FEATURES]

print(f"Feature count: {len(FEATURES)}")
print("Features:", FEATURES)

### Step 5: Impute Missing Values

In [ ]:
imputer = SimpleImputer(strategy='median')
X_train_imp = pd.DataFrame(imputer.fit_transform(X_train), columns=FEATURES)
X_pub_imp   = pd.DataFrame(imputer.transform(X_pub), columns=FEATURES)
X_priv_imp  = pd.DataFrame(imputer.transform(X_priv), columns=FEATURES)

print("Imputation complete.")
print("Training set shape:", X_train_imp.shape)

### Step 6: Cross-Validation Evaluation

We evaluate all three models using 5-fold stratified cross-validation on the training set to compare their F1 scores.

In [ ]:
xgb_model = xgb.XGBClassifier(
    n_estimators=500, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, scale_pos_weight=2.2,
    eval_metric='logloss', random_state=42, n_jobs=-1
)

lgb_model = lgb.LGBMClassifier(
    n_estimators=500, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, class_weight='balanced',
    random_state=42, n_jobs=-1, verbose=-1
)

rf_model = RandomForestClassifier(
    n_estimators=300, max_depth=8, class_weight='balanced',
    random_state=42, n_jobs=-1
)

print("5-Fold Stratified CV F1 Scores (on train set):")
print("-" * 45)
for name, model in [('XGBoost', xgb_model), ('LightGBM', lgb_model), ('Random Forest', rf_model)]:
    scores = cross_val_score(model, X_train_imp, y_train, cv=5, scoring='f1', n_jobs=-1)
    print(f"  {name:<15}: {scores.mean():.4f} ± {scores.std():.4f}")

### Step 7: Train Final Models on Train + Public Test

Since `public_test.csv` includes labels, we incorporate it into training to maximize the information available to the final models.

In [ ]:
X_full = pd.concat([X_train_imp, X_pub_imp], ignore_index=True)
y_full = pd.concat([y_train, y_pub], ignore_index=True)

print("Training on combined train + public data...")
xgb_model.fit(X_full, y_full)
lgb_model.fit(X_full, y_full)
rf_model.fit(X_full, y_full)
print("All models trained!")

# Feature importance from XGBoost
fi = pd.Series(xgb_model.feature_importances_, index=FEATURES).sort_values(ascending=False)
print("\nTop 10 features (XGBoost importance):")
print(fi.head(10))

### Step 8: Threshold Tuning on Public Test

F1 score is sensitive to the decision threshold. We sweep thresholds on the public test set (which has ground-truth labels) to find the value that maximizes F1.

In [ ]:
xgb_pub = xgb_model.predict_proba(X_pub_imp)[:, 1]
lgb_pub = lgb_model.predict_proba(X_pub_imp)[:, 1]
rf_pub  = rf_model.predict_proba(X_pub_imp)[:, 1]
pub_proba = 0.4 * xgb_pub + 0.4 * lgb_pub + 0.2 * rf_pub

best_thresh, best_f1 = 0.5, 0
for t in np.arange(0.30, 0.70, 0.01):
    preds = (pub_proba >= t).astype(int)
    f = f1_score(y_pub, preds)
    if f > best_f1:
        best_f1, best_thresh = f, t

print(f"Best threshold : {best_thresh:.2f}")
print(f"Public test F1 : {best_f1:.4f}")

print("\nClassification report on public test:")
pub_preds = (pub_proba >= best_thresh).astype(int)
print(classification_report(y_pub, pub_preds, target_names=['Not Converted', 'Converted']))

### Step 9: Generate Predictions and Create Submission

In [ ]:
xgb_priv = xgb_model.predict_proba(X_priv_imp)[:, 1]
lgb_priv = lgb_model.predict_proba(X_priv_imp)[:, 1]
rf_priv  = rf_model.predict_proba(X_priv_imp)[:, 1]
ensemble_proba = 0.4 * xgb_priv + 0.4 * lgb_priv + 0.2 * rf_priv

final_preds = (ensemble_proba >= best_thresh).astype(int)
print("Prediction distribution:", pd.Series(final_preds).value_counts().to_dict())

submission = pd.DataFrame({
    'User_ID': priv['User_ID'],
    'Converted': final_preds
})
submission.to_csv('submission.csv', index=False)
print("\nsubmission.csv created successfully!")
submission.head(10)